In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vfomenko/young-affectnet-hq")

print("Path to dataset files:", path)

Path to dataset files: /data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1


In [5]:
import os

# 指定文件夹路径
folder_path = '/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1'

# 获取文件夹下的所有子目录
subdirs = [os.path.join(folder_path,d) for d in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, d))]

# 打印子目录
print(subdirs)


['/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1/anger', '/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1/neutral', '/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1/contempt', '/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1/happy', '/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1/fear', '/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1/disgust', '/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1/surprise', '/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1/sad']


In [6]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
import requests
#from datasets import load_dataset
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
import io
from sklearn.model_selection import train_test_split
import numpy as np
import PIL
from torch.utils.data import Dataset as D
import torchvision.transforms as transforms
from tqdm import tqdm
# 示例数据
#from datasets import clear_dataset_cache
#清除缓存
#clear_dataset_cache()

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
#from datasets import load_dataset
from transformers import ViTImageProcessor, ViTForImageClassification
from sklearn.model_selection import train_test_split
import numpy as np
from PIL import Image
import csv


transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 调整为统一大小
    transforms.ToTensor()
])
id2label={
    4: "anger",
    7: "contempt",
    5: "disgust",
    3: "fear",
    1: "happy",
    0: "neutral",
    6: "sad",
    2: "surprise"}
# 假设你的字典数据如下
data = []
d=dict()
def get_key(val,d):
    
    for key, value in d.items():
        
        if value==val:
            #print(key)
            return key
features=[]
for d in tqdm(subdirs,position=0,leave=True):
    #print(d)
    for di in os.listdir(d):
        #print(os.path.join(d,di))
        image = Image.open(os.path.join(d,di))
        image=transform(image)
        #print(image.shape)#(3,224,224)
        label=d.split(os.sep)[-1]
        
        key_label=get_key(label,id2label)
        #print((label,key_label))
        features.append([image,key_label])
        #data.append({'path':dir,'label':key_label})
"""plt.imshow(image)
plt.axis('off')  # 不显示坐标轴
plt.show()"""

print(features)


100%|██████████| 8/8 [05:49<00:00, 43.69s/it]
IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [7]:
print(features[13200])

[tensor([[[0.7765, 0.8039, 0.8196,  ..., 0.5490, 0.5608, 0.5765],
         [0.8078, 0.8078, 0.8118,  ..., 0.5412, 0.5451, 0.5569],
         [0.8157, 0.8118, 0.8078,  ..., 0.5333, 0.5373, 0.5490],
         ...,
         [0.7686, 0.7569, 0.7490,  ..., 0.7255, 0.7137, 0.6745],
         [0.7686, 0.7686, 0.7647,  ..., 0.7020, 0.7216, 0.7216],
         [0.7647, 0.7608, 0.7490,  ..., 0.6941, 0.7137, 0.7412]],

        [[0.7176, 0.7451, 0.7608,  ..., 0.4784, 0.4902, 0.5059],
         [0.7490, 0.7490, 0.7529,  ..., 0.4706, 0.4745, 0.4863],
         [0.7569, 0.7529, 0.7490,  ..., 0.4627, 0.4667, 0.4784],
         ...,
         [0.7059, 0.6980, 0.6902,  ..., 0.6667, 0.6549, 0.6118],
         [0.7059, 0.7098, 0.7059,  ..., 0.6431, 0.6627, 0.6588],
         [0.7020, 0.7020, 0.6902,  ..., 0.6353, 0.6549, 0.6784]],

        [[0.6902, 0.7176, 0.7333,  ..., 0.4314, 0.4431, 0.4588],
         [0.7216, 0.7216, 0.7255,  ..., 0.4235, 0.4275, 0.4392],
         [0.7294, 0.7255, 0.7216,  ..., 0.4157, 0.4196, 0

In [8]:
class MyDataset(Dataset):
    def __init__(self, x, y):
       
        self.labels = y
        # 转置整个数据集
        self.x = x
        #print(x.shape)
        #print(self.x.shape)
        self.preprocess = transforms.Compose([
            #transforms.ToPILImage(),
            #transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = self.x[idx]
        #print(img.shape)
        # 转置回 (height, width, channels) 以适应 PIL
        img = np.transpose(img, (1, 2, 0))
        X = self.preprocess(img)
        #print(X.shape)
        return X, self.labels[idx]

In [9]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 调整为统一大小
    #transforms.ToTensor()
])
from tqdm import tqdm
# 提取特征和标签
X = []
y = []
#.shuffle(seed=40).select(range(10000))
from sklearn.preprocessing import OneHotEncoder
import numpy as np
for example in tqdm(features,position=0,leave=True):  
    
    X.append(example[0])
    y.append(example[1])
    #print(example[1])
    #print(type(example[1]))
#print(y)
X = np.array(X)
y = np.array(y)
all_categories = [np.array([0,1, 2, 3, 4, 5,6,7])]


eshaped_lst = y.reshape(-1, 1) 
#print(eshaped_lst)
# 创建 OneHotEncoder 对象，并指定类别
encoder = OneHotEncoder(categories=all_categories)
one_hot_result = encoder.fit_transform(np.array(eshaped_lst)).toarray()
# 转换为 NumPy 数组
#print(one_hot_result.toarray())
y=one_hot_result
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#print(X_train.shape)


train_dataset = MyDataset(X_train,y_train)
test_dataset = MyDataset(X_test, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(test_dataset, batch_size=32)
print("数据加载器加载完毕")
# 示例：遍历数据加载器







100%|██████████| 14648/14648 [00:00<00:00, 1018587.88it/s]


数据加载器加载完毕


In [30]:
for x, y in train_dataloader:
    
    print(f"label形状:{y.shape}")

label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([16, 8])
label形状:torch.Size([

In [13]:
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
!export CUDA_LAUNCH_BLOCKING=1

In [19]:
from torch.cuda.amp import GradScaler, autocast
from torch.nn import Linear, Conv2d, BatchNorm1d, BatchNorm2d, PReLU, ReLU, Sigmoid, Dropout2d, Dropout, AvgPool2d, MaxPool2d, AdaptiveAvgPool2d, Sequential, Module, Parameter
class Flatten(Module):
    def forward(self, input):
        return input.view(input.size(0), -1)
early_stop=50
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
import torch.nn as nn
def vit():
    # 处理器，用于将图像转换为模型所需的格式
    processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
    # 加载预训练的 ViT 模型
    model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
    for param in model.parameters():
	    param.requires_grad = True
    # 获取最后一个 ViTLayer
    last_layer = model.vit.encoder.layer[-1]
    
    # 修改最后一个 output 层的 dense 线性层维度
    last_layer.output.dense = nn.Linear(in_features=3072, out_features=512, bias=True)
    
    # 修改 layernorm_before 和 layernorm_after 的维度
    last_layer.layernorm_before = nn.LayerNorm((512,), eps=1e-12, elementwise_affine=True)
    last_layer.layernorm_after = nn.LayerNorm((512,), eps=1e-12, elementwise_affine=True)
    
    # 修改模型最后的 layernorm 层的维度
    model.vit.layernorm = nn.LayerNorm((512,), eps=1e-12, elementwise_affine=True)
    
    # 修改分类器的输入维度
    model.classifier = nn.Linear(in_features=512, out_features=8, bias=True)
 
    model.config.id2label={
    4: "anger",
    7: "contempt",
    5: "disgust",
    3: "fear",
    1: "happy",
    0: "neutral",
    6: "sad",
    2: "surprise"}
    """Anger: 4
Disgust: 5
Fear: 3
Happiness: 1
Neutral: 0
Sadness: 6
Surprise: 2
"""
    return model,processor
model,processor=vit()
print(model)
# 检查是否有多个 GPU
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)

# 将模型移动到 GPU 上
model = model.cuda()

#print(type(dataset["train"]))



# 使用 AdamW 优化器
optimizer = AdamW(model.parameters(), lr=1e-5)

# 设置学习率调度器
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)


# 损失函数
criterion = torch.nn.CrossEntropyLoss()
best_acc=0
from tqdm import tqdm
step=0
# 训练模型
for epoch in range(1000):  # 10 个 epoch
    print("Epoch:",epoch)
    model.train()
    for x,y in tqdm(train_dataloader,position=0,leave=True):
        
        #print("输入模型的x形状:",x.shape)#torch.Size([32, 224, 224, 3])->torch.Size([32, 3, 224, 224])
        # 获取输入和标签
        inputs = x.to(device)
        labels = y.to(device)
        #print(labels.shape)
        # 前向传播
        #with autocast():
        outputs = model(inputs)
        logits = outputs.logits
        # 计算损失
        loss = criterion(logits, labels)
        
        #print(logits.shape)
        #print(labels)
        #print(f"loss:{loss}")
        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # 更新学习率
    scheduler.step()
    
    # 每个 epoch 之后验证模型
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x,y in tqdm(val_dataloader,position=0,leave=True):
            inputs = x.to(device)
            labels = y.to(device)
            
            outputs = model(inputs)
            logits = outputs.logits
            #print("预测值",logits)
            
            _, predicted = torch.max(logits, dim=1)
            #print(predicted)
            #print(torch.max(labels,dim=1)[1])
            #print("真实标签:",labels)
            #print("预测类别:",predicted)
            total += labels.size(0)
            correct += (predicted == torch.max(labels,dim=1)[1]).sum().item()
            #print(correct)
    # 输出每个 epoch 的准确率
    accuracy = correct / total * 100
    print(f"Epoch {epoch+1}, Accuracy: {accuracy:.2f}%")
    if accuracy>best_acc:
        best_acc=accuracy
        model.module.save_pretrained('./vit_finetuned')
        processor.save_pretrained('./vit_finetuned')
        print(f"best_acc:{best_acc}")
        step=0
    else:
        step+=1
        if step>=early_stop:
            break
print(model)

# 加载微调后的模型
model = ViTForImageClassification.from_pretrained('./vit_finetuned')

processor = ViTImageProcessor.from_pretrained('./vit_finetuned')

# 检查是否有多个 GPU
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)

# 将模型移动到 GPU 上
model = model.cuda()
print(model.module.classifier)
# 预测
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 调整为统一大小
    transforms.ToTensor()
])

image = Image.open("/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1/sad/10_image0000093.jpg")
image=transform(image)
inputs = processor(images=image, return_tensors="pt").to(device)
outputs = model(**inputs)
logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()
print(f"Predicted class: {predicted_class_idx}")
print(f"Predicted class: {model.module.config.id2label[predicted_class_idx]}")


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-10): 11 x ViTLayer(
          (attention): ViTSdpaAttention(
            (attention): ViTSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_fe

  0%|          | 0/367 [00:00<?, ?it/s]


RuntimeError: Caught RuntimeError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/data/liuran/anaconda3/envs/test1/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py", line 96, in _worker
    output = module(*input, **kwargs)
  File "/data/liuran/anaconda3/envs/test1/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/data/liuran/anaconda3/envs/test1/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1747, in _call_impl
    return forward_call(*args, **kwargs)
  File "/data/liuran/anaconda3/envs/test1/lib/python3.10/site-packages/transformers/models/vit/modeling_vit.py", line 856, in forward
    outputs = self.vit(
  File "/data/liuran/anaconda3/envs/test1/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1736, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/data/liuran/anaconda3/envs/test1/lib/pytho

In [16]:
from transformers import ViTImageProcessor, ViTForImageClassification
import torch
# 63.62加载微调后的模型
model = ViTForImageClassification.from_pretrained('./vit_finetuned')

processor = ViTImageProcessor.from_pretrained('./vit_finetuned')

# 检查是否有多个 GPU
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)

# 将模型移动到 GPU 上
model = model.cuda()
print(model.module.classifier)
# 预测
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 调整为统一大小
    transforms.ToTensor()
])

image = Image.open("/data/liuran/.cache/kagglehub/datasets/vfomenko/young-affectnet-hq/versions/1/sad/10_image0000093.jpg")
image=transform(image)
inputs = processor(images=image, return_tensors="pt").to(device)
outputs = model(**inputs)
logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()
print(f"Predicted class: {predicted_class_idx}")
print(f"Predicted class: {model.module.config.id2label[predicted_class_idx]}")

Some weights of the model checkpoint at ./vit_finetuned were not used when initializing ViTForImageClassification: ['classifier.0.bias', 'classifier.0.weight', 'classifier.1.bias', 'classifier.1.weight', 'classifier.4.bias', 'classifier.4.weight']
- This IS expected if you are initializing ViTForImageClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ViTForImageClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of ViTForImageClassification were not initialized from the model checkpoint at ./vit_finetuned and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predict

Using 8 GPUs
Linear(in_features=768, out_features=8, bias=True)
Predicted class: 3
Predicted class: fear


In [2]:
import torch
print(torch.__version__)

2.5.1


In [ ]:
!nvidia-smi

In [3]:
!top


=top - 23:40:35 up 32 days,  3:31,  2 users,  load average: 3.29, 3.03, 3.10
Tasks: 1119 total,   4 running, 1029 sleeping,  86 stopped,   0 zombie
%Cpu(s):  4.4 us,  0.7 sy,  0.0 ni, 95.0 id,  0.0 wa,  0.0 hi,  0.0 si,  0.0 st
KiB Mem : 52807673+total, 11179856+free, 10872176+used, 30755641+buff/cache
KiB Swap: 41943036 total, 40966140 free,   976896 used. 41313113+avail Mem 

  PID USER      PR  NI    VIRT    RES    SHR S  %CPU %MEM     TIME+ COMMAND     
21496 yanglon+  20   0   28.3g   1.3g 306440 R 105.3  0.2 362:27.64 python3.11  
10207 liuran    20   0   29.5g   1.3g 470136 R 100.0  0.3   1:01.84 python      
65565 yaohail+  20   0  964136  90880  20856 R 100.0  0.0  29626:43 node        
 9268 zhangfe+  20   0  250596  53692   7280 S  26.3  0.0   0:24.17 pip         
12885 liuran    20   0  165112   3120   1560 R  21.1  0.0   0:00.09 top         
21211 yanglon+  20   0 1249540  16776   5980 S  10.5  0.0   1:13.82 clash-linu+ 
22453 yanglon+  20   0 8000868   3.8g 429976 S  10.

In [ ]:
!pip install jupyter